# Crawl a STAC catalog

Configure and run a checkpointed STAC ingestion into PostGIS. Running the crawl cell makes network requests and writes to the configured database. The crawler commits each page with its next-page checkpoint, so an interrupted run can be resumed using its run ID. Transient rejections of freshly issued pagination tokens are retried automatically.

## Configuration

Leave `run_id` as `None` to start a new run with the catalog settings below. Set it to an existing run ID to resume that run, or also set `re_crawl` to `True` to restart its saved query from the first page.

In [ ]:
catalog_url = "https://catalog.maap.eo.esa.int/catalogue/"
collections = ["BiomassLevel1c"]
cql2_filter = {}
page_size = 100

run_id = None
re_crawl = False

## Execute

The final value is a `CrawlResult`. Keep its `run_id` if the crawl is interrupted.

In [ ]:
from stac_dupes import crawler, db
from stac_dupes.config import database_url

if re_crawl and run_id is None:
    raise ValueError("re_crawl requires an existing run_id")

if run_id is None:
    crawl_options = {
        "catalog_url": catalog_url,
        "cql2_filter": cql2_filter,
        "collections": collections,
        "page_size": page_size,
    }
else:
    crawl_options = {"run_id": run_id, "re_crawl": re_crawl}

with db.connect(database_url()) as connection:
    db.apply_migrations(connection)
    result = crawler.crawl(connection, **crawl_options)

result